In [1]:
import numpy as np
import pandas as pd

import load_data
import cutpoint_analysis

import os
import pickle
import datetime
import time

import random

from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, average_precision_score, precision_recall_curve, RocCurveDisplay, accuracy_score

import mlflow
from mlflow.models import infer_signature

import great_tables as gt
from great_tables import style, loc
from great_tables import exibble

# Initialize

## Load data (both imputation versions)

`imp_v1_df` has missing values imputed with the median. `imp_v2_df` has missing lab values imputed with the latest lab value for the patient if available, and then with median if still missing.

In [2]:
imp_v1_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_normalized_includes_bSCr.csv')
imp_v2_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv')

imp_v1_df['aki_72hrs_any'] = [int(np.round(aki)) for aki in imp_v1_df['aki_72hrs_any']]
imp_v2_df['aki_72hrs_any'] = [int(np.round(aki)) for aki in imp_v2_df['aki_72hrs_any']]

## Load selected feature set dict

There are four selected feature sets - one for each combination of imputation method and metric of interest (AUROC or AUPRC). The SCr feature can either be prior or baseline, so there will be two versions of each of these feature sets for a total of 8 feature sets to test.

In [5]:
final_feature_set_dict = dict()

for filename in os.listdir('trimmed_feature_sets'):
    if '.pickle' in filename:
        key = filename.replace('.pickle', '')
        with open('trimmed_feature_sets/' + filename, 'rb') as infile:
            final_feature_set_dict[key] = pickle.load(infile)

In [6]:
# with open('../pickle/2024-02-09 - Final Feature Set Dictionary (Includes baseline_bSCr).pickle', 'rb') as infile:
#     final_feature_set_dict = pickle.load(infile)

# baseline_feature_set_dict = feature_set_dict.copy()
# prior_feature_set_dict = feature_set_dict.copy()

baseline_feature_set_dict = dict()
baseline_feature_set_intersection_dict = dict()
prior_feature_set_dict = dict()

for key in final_feature_set_dict.keys():
    baseline_feature_set_dict[key] = []
    prior_feature_set_dict[key] = []
    for feature_set in final_feature_set_dict[key]:
        for feature in feature_set:
            if feature not in baseline_feature_set_dict[key] and feature != 'bSCr_prior':
                baseline_feature_set_dict[key].append(feature)
            if feature not in prior_feature_set_dict[key] and feature != 'baseline_bSCr':
                prior_feature_set_dict[key].append(feature)

    baseline_feature_set_intersection_dict[key] = [
        feature for feature in final_feature_set_dict[key][0] if \
            all([feature in final_feature_set_dict[key][i] for i in range(len(final_feature_set_dict[key]))])
    ]
            # baseline_feature_set_dict[key] = [feature for feature in final_feature_set_dict[key] if feature != 'bSCr_prior']
            # prior_feature_set_dict[key] = [feature for feature in final_feature_set_dict[key] if feature != 'baseline_bSCr']
    # prior_feature_set_dict[key].remove('baseline_bSCr')

In [7]:
print('\n'.join(final_feature_set_dict.keys()))

auprc_ll_10_iter_logreg_only
auprc_ll_3_iter_logreg_only
auprc_med_3_iter
auprc_med_3_iter_logreg_only
auroc_ll_10_iter_logreg_only
auroc_ll_3_iter
auroc_ll_3_iter_logreg_only
auroc_med_10_iter_logreg_only
auroc_med_3_iter
auroc_med_3_iter_logreg_only


In [8]:
# for key in baseline_feature_set_intersection_dict.keys():
#     print('\n' + '='*10)
#     print(key + '\n')
#     print('\n'.join(baseline_feature_set_intersection_dict[key]))
#     print('='*10)

# Modeling

## Set tracking uri for MLFlow and start experiment so model parameters and performance can be logged.

## Define a function to assess performance of input model type using input feature set and dataset.

In [9]:
def assess_model_on_features(train_df,
                             test_df,
                             feature_set,
                             model_callable,
                             params_dict,
                             feature_set_label,
                             metric_label,
                             model_label,
                             target_feature='aki_72hrs_any',
                             random_state=343,
                             additional_param_dict=dict()):
    X_train = train_df[feature_set].to_numpy()
    X_test = test_df[feature_set].to_numpy()
    
    y_train = train_df[target_feature].to_numpy()
    y_test = test_df[target_feature].to_numpy()

    model = model_callable(**params_dict)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)

    performance_results = {
        'cutpoint_50': cutpoint_analysis.get_results_at_cutpoint(y_test, 
                                                                 y_score[:,1],
                                                                 cutpoint=0.5)['metrics'],
        'cutpoint_90': cutpoint_analysis.get_results_at_cutpoint(y_test, 
                                                                 y_score[:,1],
                                                                 cutpoint=0.9)['metrics']
    }

    full_metrics = {
        'accuracy_cp50': performance_results['cutpoint_50']['accuracy'],
        'tpr_cp50': performance_results['cutpoint_50']['tpr'],
        'tnr_cp50': performance_results['cutpoint_50']['tnr'],
        'precision_cp50': performance_results['cutpoint_50']['precision'],
        'accuracy_cp90': performance_results['cutpoint_90']['accuracy'],
        'tpr_cp90': performance_results['cutpoint_90']['tpr'],
        'tnr_cp90': performance_results['cutpoint_90']['tnr'],
        'precision_cp90': performance_results['cutpoint_90']['precision'],
        'auroc': roc_auc_score(y_test, y_score[:,1]),
        'auprc': average_precision_score(y_test, y_score[:,1])
    }

    return full_metrics

## Initialize train and test sets for each dataset.

In [10]:
train_df_imp_v1, test_df_imp_v1 = train_test_split(imp_v1_df, test_size=0.2, random_state=343, stratify=imp_v1_df['aki_72hrs_any'])
train_df_imp_v2, test_df_imp_v2 = train_test_split(imp_v2_df, test_size=0.2, random_state=343, stratify=imp_v2_df['aki_72hrs_any'])

## Define dictionaries of model callables and parameters

In [11]:
model_callable_dict = {
    'Logistic Regression': LogisticRegression,
    'Random Forest': RandomForestClassifier,
    'DecisionTree': DecisionTreeClassifier,
    'Gradient Boosting Classifier': GradientBoostingClassifier,
    'Naive Bayes (Gaussian)': GaussianNB,    
    'SVC (rbf)': SVC,
    'SVC (linear)': SVC,
    'SVC (poly)': SVC
}

model_callable_dict_no_svc = {
    'Logistic Regression': LogisticRegression,
    'Random Forest': RandomForestClassifier,
    'DecisionTree': DecisionTreeClassifier,
    'Gradient Boosting Classifier': GradientBoostingClassifier,
    'Naive Bayes (Gaussian)': GaussianNB
}

model_callable_dict_svc = {
    'SVC (rbf)': SVC,
    'SVC (linear)': SVC,
    'SVC (poly)': SVC
}

model_params_dict = {
    'Logistic Regression': {
        'class_weight': 'balanced',
        'max_iter': 10000,
        'random_state': 343
    },    
    'SVC (rbf)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'rbf'
    },
    'SVC (linear)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'linear'
    },
    'SVC (poly)': {
        'probability': True,
        'max_iter': -1,
        'class_weight': 'balanced',
        'random_state': 343,
        'kernel': 'poly'
    },
    'Random Forest': {
        'random_state': 343,
        'class_weight': 'balanced'
    },
    'DecisionTree': {
        'random_state': 343,
        'class_weight': 'balanced'
    },
    'Gradient Boosting Classifier': {
        'random_state': 343
    },
    'Naive Bayes (Gaussian)': dict()
}

In [12]:
train_df_dict = {
    'imp_v1_auprc': train_df_imp_v1,
    'imp_v1_auroc': train_df_imp_v1,
    'imp_v2_auprc': train_df_imp_v2,
    'imp_v2_auroc': train_df_imp_v2
}

test_df_dict = {
    'imp_v1_auprc': test_df_imp_v1,
    'imp_v1_auroc': test_df_imp_v1,
    'imp_v2_auprc': test_df_imp_v2,
    'imp_v2_auroc': test_df_imp_v2
}

## Train non-SVC models

In [13]:
results_dd_baseline = dict()
results_dd_baseline_intersection = dict()
results_dd_prior = dict()

for feature_set_key in baseline_feature_set_dict.keys():
    print('~'*25)
    print('='*20)
    print(feature_set_key)
    print('='*20)
    feature_set_baseline = baseline_feature_set_dict[feature_set_key]
    feature_set_baseline_intersection = baseline_feature_set_intersection_dict[feature_set_key]
    feature_set_prior = prior_feature_set_dict[feature_set_key]
    metric_label = feature_set_key.split('_')[-1]
    results_dd_baseline[feature_set_key] = dict()
    results_dd_prior[feature_set_key] = dict()
    results_dd_baseline_intersection[feature_set_key] = dict()
    
    for model_key in model_callable_dict_no_svc.keys():
        print(model_key)
        if '_ll_' in feature_set_key:
            train_df = train_df_imp_v2
            test_df = test_df_imp_v2
        else:
            train_df = train_df_imp_v1
            test_df = test_df_imp_v1
        # test_df = test_df_dict[feature_set_key]
        model_callable = model_callable_dict_no_svc[model_key]
        params_dict = model_params_dict[model_key]
        
        baseline_union_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_baseline_union.pickle'
        
        if os.path.isfile(baseline_union_filename) == False:
            print('[baseline]')
            results_dd_baseline[feature_set_key][model_key] = assess_model_on_features(
                train_df,
                test_df,
                feature_set_baseline,
                model_callable,
                params_dict,
                feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343,
                additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'union'})
            
            with open(baseline_union_filename, 'wb') as outfile:
                pickle.dump(results_dd_baseline[feature_set_key][model_key], outfile)

        baseline_intersect_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_baseline_intersection.pickle'
        
        if os.path.isfile(baseline_intersect_filename) == False:
            print('[baseline_intersection]')
            results_dd_baseline_intersection[feature_set_key][model_key] = assess_model_on_features(
                train_df,
                test_df,
                feature_set_baseline_intersection,
                model_callable,
                params_dict,
                feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343,
                additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'intersection'})
            
            with open(baseline_intersect_filename, 'wb') as outfile:
                pickle.dump(results_dd_baseline_intersection[feature_set_key][model_key], outfile)

        prior_union_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_prior_union.pickle'
        
        if os.path.isfile(prior_union_filename) == False:
            print('[prior]')
            results_dd_prior[feature_set_key][model_key] = assess_model_on_features(train_df,
                                                                             test_df,
                                                                             feature_set_prior,
                                                                             model_callable,
                                                                             params_dict,
                                                                             feature_set_prior,
                                                                             metric_label,
                                                                             model_key,
                                                                             target_feature='aki_72hrs_any',
                                                                             random_state=343,
                                                                             additional_param_dict={'bSCr': 'prior'})
            
            with open(prior_union_filename, 'wb') as outfile:
                pickle.dump(results_dd_prior[feature_set_key][model_key], outfile)

~~~~~~~~~~~~~~~~~~~~~~~~~
auprc_ll_10_iter_logreg_only
Logistic Regression
[baseline]
[baseline_intersection]
[prior]
Random Forest
[baseline]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
[baseline_intersection]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
[prior]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
DecisionTree
[baseline]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
[baseline_intersection]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
[prior]
Max number of iterations reached - returning results early.
Max number of iterations reached - returning results early.
Gradient Boosting Classifier
[baseline]
Max numbe

In [14]:
baseline_union_filename

'model_performance_results/auroc_med_3_iter_logreg_only_Naive Bayes Gaussian_baseline_union.pickle'

In [15]:
os.path.isfile(baseline_union_filename)

True

## Save results of non-SVC models

In [ ]:
with open('model_performance_results_baseline_union_no_svc.pickle', 'wb') as outfile:
    pickle.dump(results_dd_baseline, outfile)

with open('model_performance_results_baseline_intersection_no_svc.pickle', 'wb') as outfile:
    pickle.dump(results_dd_baseline_intersection, outfile)

with open('model_performance_results_prior_union_no_svc.pickle', 'wb') as outfile:
    pickle.dump(results_dd_prior, outfile)

# Train SVC models

In [17]:
results_dd_baseline_svc = dict()
results_dd_baseline_intersection_svc = dict()
results_dd_prior_svc = dict()

for feature_set_key in baseline_feature_set_dict.keys():
    print('~'*25)
    print('='*20)
    print(feature_set_key)
    print('='*20)
    feature_set_baseline = baseline_feature_set_dict[feature_set_key]
    feature_set_baseline_intersection = baseline_feature_set_intersection_dict[feature_set_key]
    feature_set_prior = prior_feature_set_dict[feature_set_key]
    metric_label = feature_set_key.split('_')[-1]
    results_dd_baseline_svc[feature_set_key] = dict()
    results_dd_prior_svc[feature_set_key] = dict()
    results_dd_baseline_intersection_svc[feature_set_key] = dict()
    
    for model_key in model_callable_dict_svc.keys():
        print(model_key)
        if '_ll_' in feature_set_key:
            train_df = train_df_imp_v2
            test_df = test_df_imp_v2
        else:
            train_df = train_df_imp_v1
            test_df = test_df_imp_v1
        # test_df = test_df_dict[feature_set_key]
        model_callable = model_callable_dict_svc[model_key]
        params_dict = model_params_dict[model_key]
        
        baseline_union_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_baseline_union.pickle'
        
        if os.path.isfile(baseline_union_filename) == False:
            print('[baseline]')
            results_dd_baseline_svc[feature_set_key][model_key] = assess_model_on_features(
                train_df,
                test_df,
                feature_set_baseline,
                model_callable,
                params_dict,
                feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343,
                additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'union'})
            
            with open(baseline_union_filename, 'wb') as outfile:
                pickle.dump(results_dd_baseline_svc[feature_set_key][model_key], outfile)

        baseline_intersect_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_baseline_intersection.pickle'
        
        if os.path.isfile(baseline_intersect_filename) == False:
            print('[baseline_intersection]')
            results_dd_baseline_intersection_svc[feature_set_key][model_key] = assess_model_on_features(
                train_df,
                test_df,
                feature_set_baseline_intersection,
                model_callable,
                params_dict,
                feature_set_key,
                metric_label,
                model_key,
                target_feature='aki_72hrs_any',
                random_state=343,
                additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'intersection'})
            
            with open(baseline_intersect_filename, 'wb') as outfile:
                pickle.dump(results_dd_baseline_intersection_svc[feature_set_key][model_key], outfile)

        prior_union_filename = 'model_performance_results/' + feature_set_key + '_' + model_key.replace('(', '').replace(')', '') + '_prior_union.pickle'
        
        if os.path.isfile(prior_union_filename) == False:
            print('[prior]')
            results_dd_prior_svc[feature_set_key][model_key] = assess_model_on_features(train_df,
                                                                             test_df,
                                                                             feature_set_prior,
                                                                             model_callable,
                                                                             params_dict,
                                                                             feature_set_prior,
                                                                             metric_label,
                                                                             model_key,
                                                                             target_feature='aki_72hrs_any',
                                                                             random_state=343,
                                                                             additional_param_dict={'bSCr': 'prior'})
            
            with open(prior_union_filename, 'wb') as outfile:
                pickle.dump(results_dd_prior_svc[feature_set_key][model_key], outfile)

~~~~~~~~~~~~~~~~~~~~~~~~~
auprc_ll_10_iter_logreg_only
SVC (rbf)
[baseline]
[baseline_intersection]
[prior]
SVC (linear)
[baseline]
[baseline_intersection]
[prior]
SVC (poly)
[baseline]
[baseline_intersection]
[prior]
~~~~~~~~~~~~~~~~~~~~~~~~~
auprc_ll_3_iter_logreg_only
SVC (rbf)
[baseline]
[baseline_intersection]
[prior]
SVC (linear)
[baseline]
[baseline_intersection]
[prior]
SVC (poly)
[baseline]
[baseline_intersection]
[prior]
~~~~~~~~~~~~~~~~~~~~~~~~~
auprc_med_3_iter
SVC (rbf)
[baseline]
[baseline_intersection]
[prior]
SVC (linear)
[baseline]
[baseline_intersection]
[prior]
SVC (poly)
[baseline]
[baseline_intersection]
[prior]
~~~~~~~~~~~~~~~~~~~~~~~~~
auprc_med_3_iter_logreg_only
SVC (rbf)
[baseline]
[baseline_intersection]
[prior]
SVC (linear)
[baseline]
[baseline_intersection]
[prior]
SVC (poly)
[baseline]
[baseline_intersection]
[prior]
~~~~~~~~~~~~~~~~~~~~~~~~~
auroc_ll_10_iter_logreg_only
SVC (rbf)
[baseline]
[baseline_intersection]
[prior]
SVC (linear)
[baseline]
[baseline

In [ ]:
results_dd_baseline = dict()
results_dd_baseline_intersection = dict()
results_dd_prior = dict()

for feature_set_key in baseline_feature_set_dict.keys():
    print('~'*25)
    print('='*20)
    print(feature_set_key)
    print('='*20)
    feature_set_baseline = baseline_feature_set_dict[feature_set_key]
    feature_set_baseline_intersection = baseline_feature_set_intersection_dict[feature_set_key]
    feature_set_prior = prior_feature_set_dict[feature_set_key]
    metric_label = feature_set_key.split('_')[-1]
    results_dd_baseline[feature_set_key] = dict()
    results_dd_prior[feature_set_key] = dict()
    results_dd_baseline_intersection[feature_set_key] = dict()
    
    for model_key in model_callable_dict.keys():
        print(model_key)
        train_df = train_df_dict[feature_set_key]
        test_df = test_df_dict[feature_set_key]
        model_callable = model_callable_dict[model_key]
        params_dict = model_params_dict[model_key]

        print('[baseline]')
        results_dd_baseline[feature_set_key][model_key] = assess_model_on_features(train_df,
                                                                         test_df,
                                                                         feature_set_baseline,
                                                                         model_callable,
                                                                         params_dict,
                                                                         feature_set_key,
                                                                         metric_label,
                                                                         model_key,
                                                                         target_feature='aki_72hrs_any',
                                                                         random_state=343,
                                                                         additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'union'})

        print('[baseline_intersection]')
        results_dd_baseline_intersection[feature_set_key][model_key] = assess_model_on_features(
            train_df,
            test_df,
            feature_set_baseline_intersection,
            model_callable,
            params_dict,
            feature_set_key,
            metric_label,
            model_key,
            target_feature='aki_72hrs_any',
            random_state=343,
            additional_param_dict={'bSCr': 'baseline', 'fs_combination': 'intersection'})
        
        print('[prior]')
        results_dd_prior[feature_set_key][model_key] = assess_model_on_features(train_df,
                                                                         test_df,
                                                                         feature_set_prior,
                                                                         model_callable,
                                                                         params_dict,
                                                                         feature_set_prior,
                                                                         metric_label,
                                                                         model_key,
                                                                         target_feature='aki_72hrs_any',
                                                                         random_state=343,
                                                                         additional_param_dict={'bSCr': 'prior'})